# KAT-VAD v5 — **Stage 0**: do VideoMAE tokens carry local anomaly evidence? (no training)

Rules: `core/docs/v5/PREREG_V5.md` **§1 (S0-1 … S0-12)**, signed before any grid existed. Plan `.project/plans/katvad-v5-mqr.md`.

One stream over the CAP-DATA tars (same path as `colab/v2/dota_cap_videomae.ipynb`):

| what | clips | stored |
|---|---|---|
| V2-S 7×7 token grid at protocol B rows (`s1[::3]`) | all 1,129 DoTA-CAP | `Thesis-V2/cache/video_grid/…/DoTA_CAP_s3_grid7/` (≈ 1.4 GB, reused by Stage 2) |
| change maps V7 / V14 / CLIP-patch + `DoTA_s1_ncc` rows | DoTA-CAP-dev (569) | `Thesis-V2/outputs/v5/v5_stage0/maps/` (≈ 80 MB) |

Gates per clip (S0-10): **G0** `fc_norm(mean tokens)` == v2's `DoTA_CAP_s1_squash` (cos ≥ 0.9999), **G1** CLIP CLS
projection == `image_embeds` + pixel gate vs `DoTA_s1_ncc`, **G2** CAP frame aspect ≈ 16:9. No DoTA-CAP-eval label is read.

GPU work is tens of minutes; **reading the tars from Drive is most of the wall time**. Runtime: any GPU (T4 is enough),
`/content` free ≥ 3 × 8 GB. Resumable: re-run step 2 after a disconnect — finished clips are skipped.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'      # read-only here: v2 caches
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload the v5 code there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V5_STAGE0_DIRNAME'), 'this upload predates v5 Stage 0 -- re-upload the code'

ENC      = constants.V5_STAGE0_ENCODER
GROUPS   = ['1-10', '11', '12-42', '43', '44-62']
BATCH_GB = constants.MMAU_STREAM_BATCH_GB
RAW      = constants.DATA_ROOT / 'MMAU' / 'raw'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
V2_CACHE = constants.VIDEO_CACHE_DIR / ENC / f'{constants.DOTA_CAP_DATASET}_s1_squash'
DCAP     = constants.OUTPUT_ROOT / 'REPORTS' / 'dota_cap'
ALIGN    = DCAP / 'dota_cap_alignment.json'
FINAL    = DCAP / 'dota_cap_readout.json'
OUT      = constants.OUTPUT_ROOT / constants.V5_OUTPUT_DIRNAME / constants.V5_STAGE0_DIRNAME
GRID_DIR = Path(DRIVE_V2) / 'cache' / constants.V5_GRID_CACHE_DIRNAME / ENC / constants.V5_GRID_DATASET_TAG
LOCAL    = Path('/content/v5s0')
for p in (OUT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name, path in (('DoTA s1', DOTA_S1), ('V2-S DoTA-CAP s1', V2_CACHE), ('alignment', ALIGN),
                   ('final ids', FINAL), ('CAP annotation', RAW / 'cap_text_annotations.xls'),
                   ('CAP chunks', RAW / 'CAP-DATA_chunks')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:18s} {path}   OK')
print('grid cache ->', GRID_DIR)
print('maps/report ->', OUT)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu xlrd
python -c "import av, einops, faiss, xlrd; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 2 needs one'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time


def stream(cmd, name):
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print(''.join(tail))
        raise RuntimeError(f'{name} exited {proc.returncode}')

## 1. GPU throughput probe (≈ 1 min)

Times V2-S `forward_tokens` on random windows and prints the GPU-only estimate for ≈ 37 k windows
(1,129 clips at stride 3). Tar reading comes on top of this.

In [ ]:
import torch

from core.models.videomae_v2 import load_pretrained

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
enc = load_pretrained(ENC, dev)
x = torch.randn(32, 3, constants.VIDEOMAE_CLIP_FRAMES, constants.CROP_SIZE, constants.CROP_SIZE, device=dev)
with torch.no_grad():
    enc.forward_tokens(x)                      # warm-up
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(5):
        enc.forward_tokens(x)
    if dev.type == 'cuda':
        torch.cuda.synchronize()
rate = 5 * len(x) / (time.time() - t0)
print(f'{rate:.0f} windows/s on {dev} -> ~{37_000 / rate / 60:.0f} min of V2-S for 1,129 clips')
del enc, x
torch.cuda.empty_cache()

## 2. Extract per CAP group — GPU

Same grouping as v2's DoTA-CAP notebook: only the CAP videos of finalized DoTA-CAP clips are written, and the tool
**stops reading a group's tar once its last needed video is closed**. One report per group in `OUT/`.

In [ ]:
import pandas as pd

ann = pd.read_excel(RAW / 'cap_text_annotations.xls', sheet_name='annotation file')
type_of = {f'{v:06d}': int(t) for v, t in zip(ann['video'].astype(int), ann['type'].astype(int))}
alignment = json.loads(ALIGN.read_text())
final_ids = set(json.loads(FINAL.read_text())['ids'])
need_cap = {alignment['clips'][d]['cap_id'] for d in final_ids}
assert len(final_ids) == 1129, len(final_ids)
assert need_cap <= type_of.keys(), sorted(need_cap - type_of.keys())[:5]

free_gb = shutil.disk_usage('/content').free / 1e9
assert free_gb >= 3 * BATCH_GB, f'{free_gb:.0f} GB free on /content, need {3 * BATCH_GB:.0f}'

for g in GROUPS:
    lo, hi = (int(x) for x in (g.split('-') * 2)[:2])
    ids = sorted(c for c in need_cap if lo <= type_of[c] <= hi)
    print(f'group {g}: {len(ids)} CAP videos needed')
    if not ids:
        continue
    ids_file = LOCAL / f'cap_ids_{g}.txt'
    ids_file.write_text(''.join(f'{i}\n' for i in ids))
    parts = sorted((RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
    assert parts, f'no parts for group {g}'
    t0 = time.time()
    stream([sys.executable, '-m', 'core.tools.v5_stage0', 'extract',
            '--alignment', ALIGN, '--final-readout', FINAL, '--parts', *parts,
            '--work-dir', LOCAL / 'work' / g, '--dota-clip-dir', DOTA_S1, '--v2-cache', V2_CACHE,
            '--report', OUT / f'extract_{g}.json', '--out-dir', OUT, '--grid-dir', GRID_DIR,
            '--cap-ids-file', ids_file, '--batch-size', '32', '--max-batch-gb', str(BATCH_GB),
            '--device', 'auto'], 'v5_stage0 extract')
    print(f'group {g}: {(time.time() - t0) / 60:.0f} min')

## 3. Coverage check

In [ ]:
from collections import Counter

report = {}
for p in sorted(OUT.glob('extract_*.json')):
    report |= json.loads(p.read_text())
print('clips reported:', len(report), 'of', len(final_ids))
print('reasons:', Counter(e['reason'] for e in report.values()))
print('dev maps:', len(list((OUT / 'maps').glob('*.npz'))), '(expect 569)')
print('grids:', len(list(GRID_DIR.glob('*.npy'))), '(expect 1129)')
g0 = [e['g0_min_cos'] for e in report.values() if 'g0_min_cos' in e]
print(f'G0 min cos over clips: {min(g0):.6f}' if g0 else 'no G0 read')
bad = {k: v['reason'] for k, v in report.items() if v['reason'] != 'ok'}
print('failed:', bad if len(bad) < 30 else f'{len(bad)} clips')

## 4. Read-out (S0-11) — CPU

Needs the DoTA box annotations (downloaded from `MoonBlvd/Detection-of-Traffic-Anomaly` if not on Drive) and DoTA's
`metadata_val.json`. The decision is mechanical (GO-7 / GO-14 / PARTIAL / KILL; INVALID if any S0-10 gate failed).

In [ ]:
%%bash
set -e
mkdir -p /content/v5s0/dota && cd /content/v5s0/dota
[ -d annotations ] || { curl -sSL -o ann.zip https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip && unzip -q ann.zip && rm ann.zip; }
ls annotations | wc -l

In [ ]:
META = constants.DATA_ROOT / 'DoTA' / 'metadata_val.json'
if not META.exists():
    META = LOCAL / 'dota' / 'metadata_val.json'
    subprocess.run(['curl', '-sSL', '-o', str(META),
                    'https://raw.githubusercontent.com/MoonBlvd/Detection-of-Traffic-Anomaly/master/dataset/metadata_val.json'],
                   check=True)
SPLIT = LOCAL / 'dota' / 'val_ids.txt'
SPLIT.write_text(''.join(f'{v}\n' for v in json.loads(META.read_text())))
stream([sys.executable, '-m', 'core.tools.v5_stage0', 'readout',
        '--reports', *sorted(OUT.glob('extract_*.json')), '--maps-dir', OUT / 'maps',
        '--annotations-dir', LOCAL / 'dota' / 'annotations', '--metadata', META,
        '--split-file', SPLIT, '--out-dir', OUT / 'REPORTS'], 'v5_stage0 readout')
print((OUT / 'REPORTS' / 'stage0_readout.md').read_text())

## 5. Record

Paste `Thesis-V2/outputs/v5/v5_stage0/REPORTS/stage0_readout.md` back to Claude, and bring
`outputs/v5/v5_stage0/{REPORTS,extract_*.json,maps}` to the repo (the maps are small) so the read-out can be re-run
locally. Do not change anything in §1 after seeing the number — a defect becomes an amendment.

In [ ]:
drive.flush_and_unmount()
print('Drive flushed')